In [2]:
%pip install neo4j
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import os
import requests
import time
from tqdm import tqdm
from neo4j import GraphDatabase

In [4]:
class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response

In [22]:
def readlines(infile):
  with open(infile) as file:
    return file.readlines()
  


def getPath(fileName):
  return os.path.join(os.path.dirname(__file__), fileName)


def accessAlbum(albumName, albumArtist):
  url = "https://musicbrainz.org/ws/2/release"
  params = {
    "query": f'release:"{albumName}" AND artist:"{albumArtist}"',
    "fmt": "json",
    "limit": 1
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}
  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  print(data)

  if not data["releases"]:
        print(f"No results found for {albumName} by {albumArtist}")
        return None

  return data["releases"][0]["id"]
  

def accessSongs(albumId):
  url = f"https://musicbrainz.org/ws/2/release/{albumId}"
  params = {
    "inc": "recordings",
    "fmt": "json"
  }

  headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}

  response = requests.get(url, params=params, headers=headers)
  data = response.json()
  
  tracks = []
  for medium in data["media"]:
    for track in medium["tracks"]:
      # time.sleep(1)
      # genres = get_song_genres(track["recording"]["id"])
      tracks.append({
        "title": track["title"],
        "duration": track.get("length", None),
        "mbid": track["recording"]["id"]
      })
      
  return tracks


# def get_song_genres(song_mbid):
#     url = f"https://musicbrainz.org/ws/2/recording/{song_mbid}"
#     params = {
#         "inc": "genres",
#         "fmt": "json"
#     }
#     headers = {"User-Agent": "MyWebsite/1.0 (myemail@gmail.com)"}
    
#     response = requests.get(url, params=params, headers=headers)
#     data = response.json()
#     return data
    # genres = [genre["name"] for genre in data["recording"].get("genres", [])]
    # return genres


def addSongsToDB(albumName, albumYear, songs, connection):
  for song in songs:

    addSongQuery = """
MATCH (al:Album {name: $albumName, year: $albumYear})
MERGE (s:Song {name: $title, song_length: $duration, mbid: $mbid})
MERGE (al)-[:Lists]->(s)
"""

    connection.query(addSongQuery, parameters={
      "albumName": albumName, 
      "albumYear": albumYear, 
      "title":song["title"],
      "duration": song["duration"], 
      "mbid": song["mbid"]})




def replaceNodeNames(oldAlbumName, oldArtist, albumName, albumArtist, conn):
    print(f"Trying to update album: '{oldAlbumName}' -> '{albumName}'")
    print(f"Trying to update artist: '{oldArtist}' -> '{albumArtist}'")

    updateAlbumNameQuery = """
MATCH (al:Album {name: $oldAlbumName})
SET al.name = $albumName
RETURN al
"""
    result = conn.query(updateAlbumNameQuery, parameters={
       "oldAlbumName": oldAlbumName,
       "albumName":albumName,
    })
    print(f"Album update result: {result}")

    updateArtistNameQuery = """
MATCH (ar:Artist{name:$oldArtist})
SET ar.name = $albumArtist
RETURN ar
"""

    result = conn.query(updateArtistNameQuery, parameters={
       "oldArtist": oldArtist,
       "albumArtist": albumArtist
    })
    print(f"Artist update result: {result}")



def getAlbumInfo(albumId):
    url = f"https://musicbrainz.org/ws/2/release/{albumId}"
    params = {
        "inc": "artist-credits",
        "fmt": "json"
    }
    headers = {"User-Agent": "1001_Albums_Recommender/1.0 (roxyknit2000@gmail.com)"}
    
    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    print(data)
    
    albumName = data["title"]
    artist = data["artist-credit"][0]["artist"]["name"]
    
    return albumName, artist


def deleteAlbumNode(albumName, conn):
   deleteAlbumQuery = """
MATCH (al:Album {name: $albumName})
DETACH DELETE al
"""
   conn.query(deleteAlbumQuery, parameters={"albumName": albumName})
   print(f"{albumName} deleted from db")


In [24]:
# -----------------------------------------------------------------------------
# Adding albums, artists, and songs to Neo4j DB
# -----------------------------------------------------------------------------

with open("config.json") as configFile:
  config = json.load(configFile)

conn= Neo4jConnection(uri= config["uriNeo4j"], user=config["userNeo4j"], pwd=config["passwordNeo4j"])


with open("1001-albums-you-must-hear-before-you-die.csv") as file:
  albumLines = file.readlines()


# for line in albumLines[1:]:
#   line_info = line.strip().split("|")
#   addRelationshipQuery = """
# MERGE (al:Album {name: $albumName, year: $year})
# MERGE(ar:Artist {name: $artistName})
# MERGE (ar)-[ :RELEASED]->(al)
# """

#   conn.query(addRelationshipQuery, parameters={
#     "albumName": line_info[1], "year": int(line_info[0]), "artistName": line_info[2]
#     })
  

# Initial pass through lines in album
# with open("unsearchable.txt", "w") as file:
#   for line in albumLines[1:]:
#     line_info = line.strip().split("|")
#     albumName, albumArtist, albumYear = line_info[1], line_info[2], line_info[0]
#     print(albumName)
#     albumId = accessAlbum(albumName, albumArtist)
#     print(albumId)
#     if not albumId:
#       file.write(f"{albumArtist}|{albumName}|{albumYear}\n")
#       continue
#     time.sleep(1)
#     songs = accessSongs(albumId)
#     print(songs)
#     addSongsToDB(albumName, int(albumYear), songs, conn)


# Update nodes with underscores and add songs for those albums
# with open("unsearchable.txt") as file:
#   notFoundLines = file.readlines()

# with open("trial_run.txt", "w") as file:
#   for line in notFoundLines:
#     if not "_" in line:
#       file.write(line)
#       continue
    
#     oldArtist, oldName, oldYear = line.strip().split("|")
#     albumArtist, albumName, albumYear = line.replace("_", "'").strip().split("|")
#     albumId = accessAlbum(albumName, albumArtist)
#     print(albumId)
#     if not albumId:
#       file.write(line)
#       continue
#     replaceNodeNames(oldName, oldArtist, albumName, albumArtist, conn)
#     time.sleep(1)
#     songs = accessSongs(albumId)
#     print(songs)
#     addSongsToDB(albumName, int(albumYear), songs, conn)


with open("trial_run.txt") as unsearchable, open("narrowedAlbumIds.txt") as idFile:
  unsearchableLines = unsearchable.readlines()
  albumIds = idFile.readlines()

# Add remaining albums and update the node names with the albumIDs directly
with open("leftoverAlbums.txt", "w") as file:
  for oldAlbumInfo, albumId in zip(unsearchableLines, albumIds):
    if not oldAlbumInfo.strip():
      break
    oldArtist, oldName, oldYear = oldAlbumInfo.strip().split("|")
    print(oldName)
    if "NOT AVAILABLE" in albumId:
      deleteAlbumNode(oldName, conn)
      file.write(oldAlbumInfo)
      continue
    
    albumId = albumId.strip()
    newAlbum, newArtist = getAlbumInfo(albumId)
    replaceNodeNames(oldName, oldArtist, newAlbum, newArtist, conn)
    time.sleep(1)
    songs = accessSongs(albumId)
    print(songs)
    addSongsToDB(newAlbum, int(oldYear), songs, conn)


print("Done!")

Ellington at Newport _56
{'status': 'Official', 'country': 'CA', 'title': 'At Newport', 'cover-art-archive': {'artwork': True, 'back': True, 'darkened': False, 'front': True, 'count': 4}, 'asin': None, 'barcode': None, 'release-events': [{'area': {'type': None, 'iso-3166-1-codes': ['CA'], 'sort-name': 'Canada', 'disambiguation': '', 'name': 'Canada', 'id': '71bbafaa-e825-3e15-8ca9-017dcad1748b', 'type-id': None}, 'date': '1956'}], 'status-id': '4e304316-386d-3409-af2e-78857eec5cfe', 'disambiguation': '', 'packaging': None, 'id': '265348ab-2b3a-4350-99a4-a110db992b63', 'artist-credit': [{'name': 'Duke Ellington', 'artist': {'type': 'Person', 'sort-name': 'Ellington, Duke', 'name': 'Duke Ellington', 'country': 'US', 'disambiguation': 'US composer, pianist & jazz bandleader', 'id': '3af06bc4-68ad-4cae-bb7a-7eeeb45e411f', 'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df'}, 'joinphrase': ' / '}, {'artist': {'type': 'Group', 'country': None, 'name': 'Buck Clayton and His All-Stars', 'disambi